In [9]:
from pathlib import Path
import random
print(Path.cwd())


d:\programs\Transformer-translate\data


In [10]:
# 获取当前脚本所在的文件夹
folder = Path.cwd()

# 找到中文文件和英文文件
zh_path = folder/"UNPC.en-zh.zh"
en_path = folder/"UNPC.en-zh.en"
  
with zh_path.open("r", encoding="utf-8") as zh_file, \
     en_path.open("r", encoding="utf-8") as en_file:
    count_row_en = sum(1 for _ in en_file)
    count_row_zh = sum(1 for _ in zh_file)
    print(f"中文文件总行数：{count_row_zh}")
    print(f"英文文件总行数：{count_row_en}")

中文文件总行数：17451549
英文文件总行数：17451549


In [11]:
sample_num = 500000
random.seed(22)
sample_indices = set(random.sample(range(count_row_zh),sample_num))
output_path_zh = folder/"sample_zh.txt"
output_path_en = folder/"sample_en.txt"
with output_path_zh.open("w",encoding="utf-8") as f,\
    output_path_en.open("w",encoding='utf-8') as g,\
     en_path.open("r",encoding="utf-8") as en_file,\
     zh_path.open("r",encoding="utf-8") as zh_file:
    for i, (line1,line2) in enumerate(zip(zh_file,en_file)):
        if i in sample_indices:
            line1 = line1.rstrip("\r\n")
            line2 = line2.rstrip("\r\n")
            f.write(line1 + "\n")
            g.write(line2 + "\n")


In [12]:
with output_path_zh.open("r", encoding="utf-8") as zh_file, \
     output_path_en.open("r", encoding="utf-8") as en_file:
    count_row_output_en = sum(1 for _ in en_file)
    count_row_output_zh = sum(1 for _ in zh_file)
    print(f"中文文件总行数：{count_row_output_zh}")
    print(f"英文文件总行数：{count_row_output_en}")

中文文件总行数：500000
英文文件总行数：500000


In [13]:
with output_path_zh.open("r", encoding="utf-8") as zh_file, \
     output_path_en.open("r", encoding="utf-8") as en_file:
    sample_list = [ count_row_output_zh//4 * i for i in range(5)]
    for i, (zh, en) in enumerate(zip(zh_file, en_file)):
        if i in sample_list:
            print(f"\n第{i + 1}行,中文：{zh.rstrip()}, 英文：{en.rstrip()}")



第1行,中文：2. 欣见秘书长1994年5月13日的报告(S/1994/565)，, 英文：2. Welcomes the report of the Secretary-General dated 13 May 1994 (S/1994/565);

第125001行,中文：3. 科长谈到了向委员会提供服务的问题，她强调应以有序的方式协助委员会处理每届会议收到的大量文件。, 英文：3. Regarding the servicing of the Committee, the Chief of Section underlined the need to assist the Committee in an organized manner in handling the heavy documentation before it at each session.

第250001行,中文：行政和预算问题, 英文：Administrative and budgetary questions

第375001行,中文：根据决议草案第18段，设想筹备委员会将在纽约举行一届为期五天的会议，时间暂定于2012年3月，届时将有10次会议需要所有六种正式语文的双向口译服务，还需要将共计135 000字的62份文件译成所有六种正式语文。, 英文：Pursuant to paragraph 18 of the draft resolution, it is envisaged that the preparatory committee will hold one session in New York for five days, tentatively in March 2012, thus requiring interpretation services from and into the six official languages for 10 meetings and translation into the six official languages of 62 documents totalling 135,000 words.


##### 划分数据集为训练集、验证集和测试集

In [14]:
pairs = []
with output_path_en.open("r", encoding="utf-8") as en_file,\
     output_path_zh.open("r", encoding="utf-8") as zh_file:
    for i, (en,zh ) in enumerate(zip(en_file, zh_file)):
        pairs.append((en.rstrip("\r\n"), zh.rstrip("\r\n")))
random.Random(22).shuffle(pairs)

storage_dir = folder/"sample_for_train_val_test"
storage_dir.mkdir(parents=True,exist_ok=True)

output_train_zh = storage_dir/"train_zh.txt"
output_train_en = storage_dir/"train_en.txt"
output_val_zh = storage_dir/"val_zh.txt"
output_val_en = storage_dir/"val_en.txt"
output_test_zh = storage_dir/"test_zh.txt"
output_test_en = storage_dir/"test_en.txt"

with output_train_zh.open("w", encoding="utf-8") as train_zh_file, \
     output_train_en.open("w", encoding="utf-8") as train_en_file, \
     output_val_zh.open("w", encoding="utf-8") as val_zh_file, \
     output_val_en.open("w", encoding="utf-8") as val_en_file, \
     output_test_zh.open("w", encoding="utf-8") as test_zh_file, \
     output_test_en.open("w", encoding="utf-8") as test_en_file:
    for i,item in enumerate(pairs):
        if i <int(0.9*len(pairs)):
            train_en_file.write(item[0] + "\n")
            train_zh_file.write(item[1] + "\n")
        elif i<int(0.95*len(pairs)):
            val_en_file.write(item[0] + "\n")
            val_zh_file.write(item[1] + "\n")
        else:
            test_en_file.write(item[0] + "\n")
            test_zh_file.write(item[1] + "\n")